# 01 — Fetch a Paper by DOI via OpenAlex

**OpenAlex** is a fully open catalogue of the global research system. It indexes over 250 million scholarly works and provides a free REST API — no API key required.

This notebook shows how to:
1. Load the example paper dataset
2. Fetch full metadata for a single paper using its DOI
3. Explore the response fields
4. Extract the most useful fields into a flat dictionary

**API docs:** https://docs.openalex.org/api-entities/works/get-a-single-work

In [ ]:
import requests
import pandas as pd
import json

## 1. Load example papers

In [ ]:
df = pd.read_csv('../data/example_papers.csv')
print(f'Loaded {len(df)} papers')
df[['Paper id', 'DOI', 'Title', 'Inclusion decision']].head(10)

## 2. Fetch one paper from OpenAlex by DOI

OpenAlex identifies works by DOI using the URL pattern:
```
https://api.openalex.org/works/https://doi.org/<DOI>
```

It is good practice to pass a `mailto` parameter so OpenAlex can contact you if your requests cause issues (this also grants you access to the faster "polite" pool).

In [ ]:
MAILTO = 'your.email@example.com'  # replace with your e-mail

# Pick the first included paper
example_doi = df[df['Inclusion decision'] == 'include']['DOI'].iloc[0]
print('DOI:', example_doi)

In [ ]:
url = f'https://api.openalex.org/works/https://doi.org/{example_doi}'
params = {'mailto': MAILTO}

response = requests.get(url, params=params)
print('Status code:', response.status_code)

work = response.json()
print('OpenAlex ID:', work.get('id'))

## 3. Explore the raw response

In [ ]:
# Top-level keys in the response
print('Top-level keys:')
for key in work.keys():
    val = work[key]
    val_preview = str(val)[:80] if val is not None else 'None'
    print(f'  {key}: {val_preview}')

In [ ]:
# Pretty-print a specific sub-object to understand its structure
print('Open access info:')
print(json.dumps(work.get('open_access'), indent=2))

In [ ]:
print('Author list (first 3):')
print(json.dumps(work.get('authorships', [])[:3], indent=2))

In [ ]:
print('Concepts (topics) assigned by OpenAlex:')
print(json.dumps(work.get('concepts', [])[:5], indent=2))

## 4. Extract key fields into a flat dictionary

In [ ]:
def extract_work_fields(work: dict) -> dict:
    """Extract the most useful fields from an OpenAlex work object."""
    # Abstract: OpenAlex stores abstracts as an inverted index
    abstract = None
    if work.get('abstract_inverted_index'):
        index = work['abstract_inverted_index']
        words = [''] * (max(pos for positions in index.values() for pos in positions) + 1)
        for word, positions in index.items():
            for pos in positions:
                words[pos] = word
        abstract = ' '.join(words)

    authorships = work.get('authorships', [])
    authors = '; '.join(
        a['author']['display_name']
        for a in authorships
        if a.get('author')
    )
    institutions = '; '.join(
        inst['display_name']
        for a in authorships
        for inst in a.get('institutions', [])
        if inst.get('display_name')
    )
    top_concepts = ', '.join(
        c['display_name']
        for c in sorted(work.get('concepts', []), key=lambda x: -x.get('score', 0))[:5]
    )

    return {
        'openalex_id': work.get('id'),
        'doi': work.get('doi'),
        'title': work.get('title'),
        'publication_year': work.get('publication_year'),
        'publication_date': work.get('publication_date'),
        'type': work.get('type'),
        'cited_by_count': work.get('cited_by_count'),
        'is_oa': work.get('open_access', {}).get('is_oa'),
        'oa_status': work.get('open_access', {}).get('oa_status'),
        'authors': authors,
        'institutions': institutions,
        'concepts': top_concepts,
        'abstract': abstract,
    }


extracted = extract_work_fields(work)
for k, v in extracted.items():
    val = str(v)[:120] if v else v
    print(f'{k:25s}: {val}')

In [ ]:
# Display as a single-row DataFrame
pd.DataFrame([extracted]).T.rename(columns={0: 'value'})

## 5. Compare OpenAlex citation count vs. Dimensions export

Our example CSV includes `Times cited` from Dimensions. Let's compare.

In [ ]:
row = df[df['DOI'] == example_doi].iloc[0]
print(f"Title          : {row['Title']}")
print(f"Dimensions cit.: {row['Times cited']}")
print(f"OpenAlex cit.  : {extracted['cited_by_count']}")